# data incident responder: evaluation run

Everything runs in this notebook: the model is served by vLLM on the GPU, and the warehouse, dbt, the MCP server
and the agents run next to it. Runtime: A100 (80 GB) with high RAM.

Optional: add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` as Colab secrets to trace every run.

Results are written to Drive after each incident. If the runtime drops, run all again and finished incidents are skipped.

In [9]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/arihaaant/data-incident-responder.git /content/responder
%cd /content/responder
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
fatal: destination path '/content/responder' already exists and is not an empty directory.
/content/responder
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-mcp-adapters 0.3.2 requires mcp<2.0.0,>=1.24.0, but you have mcp 2.3.0 which is incompatible.
responder 0.1.0 requires mcp[cli]<2,>=1.24, but you have mcp 2.3.0 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for responder (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [10]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

tracing on


In [14]:
!git -C /content/responder pull && pip install -q -e /content/responder

remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 27 (delta 12), reused 22 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (27/27), 33.33 KiB | 1.96 MiB/s, done.
From https://github.com/arihaaant/data-incident-responder
   0e37442..3a8c789  main       -> origin/main
Updating 0e37442..3a8c789
Fast-forward
 notebooks/evaluation_run.ipynb |  6 ++--
 pyproject.toml                 |  2 +-
 src/responder/agents.py        | 67 ++++++++++++++++++++++++++++++++++--------
 src/responder/tools.py         |  7 +++--
 tests/test_agents.py           | 60 +++++++++++++++++++++++++++++++++++++
 5 files changed, 123 insertions(+), 19 deletions(-)
 create mode 100644 tests/test_agents.py
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   

## serve the model

In [15]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

model up after 0.0 min


## warehouse

In [16]:
import subprocess, sys
!python -m responder.warehouse --path kit/source.duckdb
# stop here if anything is broken, so a bad environment never produces scorecards
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout[-1500:])
assert tests.returncode == 0, "tests failed, fix before running"

built kit/source.duckdb
.....................................                                    [100%]
37 passed in 14.13s



## runs: the responder and both baselines on the 48 development incidents

In [17]:
from google.colab import drive

drive.mount("/content/drive")
run_name = "run2"  # a new name for every evaluation, since finished incidents in a folder are skipped
out = f"/content/drive/MyDrive/data-incident-responder/{run_name}"
os.makedirs(out, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.run --system {system} --split dev --workers 6 --out {out}/dev-{system}


=== multi ===
[10/10/26 06:49:25] INFO     Processing request of type            ]8;id=927144;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=539479;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/10/26 06:49:25] INFO     Processing request of type            ]8;id=204210;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=537318;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/10/26 06:49:25] INFO     Processing request of type            ]8;id=121910;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=588722;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/

## results

In [19]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.report {out}/dev-{system}


=== multi ===
                   n    root   cause  action  resolved  tokens
overall           48   87.5%   93.8%   85.4%     83.3%   16170
duplicates        12  100.0%  100.0%  100.0%    100.0%   18923
late_data         12  100.0%  100.0%  100.0%    100.0%   21951
null_spike        12   66.7%   75.0%   75.0%     75.0%   15002
schema_drift      12   83.3%  100.0%   66.7%     58.3%    8803
dev               48   87.5%   93.8%   85.4%     83.3%   16170

resolved 40/48 (95% CI 70 to 91%), fixes passing the hidden check 27/34

=== single ===
                   n    root   cause  action  resolved  tokens
overall           48   60.4%   77.1%   87.5%     77.1%   17577
duplicates        12   83.3%  100.0%   91.7%     66.7%   20283
late_data         12   50.0%   83.3%   83.3%     75.0%   24626
null_spike        12   75.0%   91.7%   91.7%     91.7%   14633
schema_drift      12   33.3%   33.3%   83.3%     75.0%   10766
dev               48   60.4%   77.1%   87.5%     77.1%   17577

resolved 37/4